# Knowledge Base node

![Knowledge base flow](rag_kb_flow.png)

A **Knowledge Base** node retrieves matching chunks from a knowledge base and has a model write an answer grounded in them: retrieval-augmented generation (RAG) as one node in a flow.

This notebook builds the knowledge base for you on **Amazon S3 Vectors**, from the three files in `sample_docs/`. They are policies for Octank Outfitters, a made-up outdoor gear store, so any answer about them has to come from the documents and not from the model's training data.

> **Teaching/Learning Tip:** the Flows knowledge base node does not support *managed* knowledge bases (the invoke fails with "This operation is not supported for managed knowledge bases"). It needs one backed by a vector store you own. S3 Vectors is the cheapest option and has nothing to provision.

> **Cost and time:** setup takes a few minutes, mostly the ingestion job. You pay for the embedding calls, the vector storage, and the model calls, all small for three short documents. The Cleanup cell deletes everything.

## Setup

In [ ]:
import sys
sys.path.append("..")  # flow_utils.py lives one folder up
import flow_utils

session, region, account = flow_utils.get_session_info()  # from your current AWS login
agent = session.client("bedrock-agent")            # control plane: flows and knowledge bases
runtime = session.client("bedrock-agent-runtime")  # data plane: invoke flows
iam = session.client("iam")
model_id = flow_utils.default_model_id(region, account)
print(f"Account {account} | region {region} | model {model_id}")

role_arn = flow_utils.ensure_role(iam, region, account)

## Build the knowledge base

`create_sample_knowledge_base` does six things, printing each as it goes:

1. Creates a private **S3 bucket** and uploads the documents.
2. Creates an **S3 vector bucket** and a **vector index** with 1024 dimensions, the size of the Titan Text Embeddings V2 vectors.
3. Creates the knowledge base's **service role**: read the bucket, call the embedding model, and read and write the index.
4. Creates the **knowledge base**, pointing it at the embedding model and the index.
5. Adds the S3 bucket as a **data source**.
6. Runs an **ingestion job**: Bedrock splits each document into chunks, embeds each chunk, and writes the vectors to the index.

> **Teaching/Learning Tip:** the index is created with `AMAZON_BEDROCK_TEXT` as a *non-filterable* metadata key. Bedrock stores each chunk's text there, and non-filterable keys may hold more data than filterable ones. This follows the AWS guidance for using S3 Vectors with Bedrock Knowledge Bases.

> **Teaching/Learning Tip:** there are now two roles. The **knowledge base** role is used while ingesting (S3, the embedding model, the index). The **flow** role only needs to query the knowledge base (`bedrock:Retrieve`, `bedrock:RetrieveAndGenerate`) and invoke the answer model.

In [ ]:
kb_state = {}  # every resource is recorded here as it is created, so cleanup works even after a failure
kb_id = flow_utils.create_sample_knowledge_base(session, region, account, "sample_docs", kb_state)

## Create the flow

Two placeholders: the knowledge base ID and the model that writes the answer from the retrieved chunks.

In [ ]:
template = flow_utils.load_template("rag_kb_flow.json", {
    "$$KNOWLEDGEBASE_ID": kb_id,
    "$$KNOWLEDGEBASE_MODEL_ID": model_id,
})

flow_id = flow_utils.create_flow(
    agent, template["name"], template["description"], template["definition"], role_arn, template.get("tags"))
version, alias_id = flow_utils.deploy_flow(agent, flow_id)
print("flow_id:", flow_id, "| version:", version, "| alias_id:", alias_id)

## Ask questions the documents answer

Each answer comes from a specific document: the Trail Tested promise in `returns-and-refunds.md`, freight shipping in `shipping.md`, and apparel warranties in `warranty-and-repairs.md`. Try a question the documents don't cover too, and see how the answer changes.

In [ ]:
for question in [
    "Can I return hiking boots I've already worn on a trail?",
    "How much does it cost to ship a kayak?",
    "What is the warranty on an Octank jacket?",
]:
    outputs = flow_utils.invoke_flow(runtime, flow_id, alias_id, question)
    print(f"Q: {question}\nA: {outputs[0]}\n")

## Cleanup

Delete the flow, then the knowledge base resources in dependency order: data source, knowledge base, vector index, vector bucket, and the S3 bucket with the documents.
The sample IAM roles are left in place. Delete them with `flow_utils.delete_role(iam)` and `flow_utils.delete_kb_role(iam)` when you are completely done.

In [ ]:
flow_utils.cleanup_flow(agent, flow_id, alias_id, version)
flow_utils.delete_sample_knowledge_base(session, kb_state)